# Company Calibration Analysis: Deterministic Scorer vs. Blind OpenAI Labels

This notebook is the empirical calibration artifact for comparing the deterministic page scorer with independent company-level OpenAI judgments. It analyzes the broad `pilot_v0_2` sample and the ÖNACE-enriched `ai_enriched_v1` sample.

The scorer outputs are unbounded lexical evidence scores, **not probabilities**. The enriched KEEP share is **not** an estimate of Austrian AI prevalence. All thresholds below are exploratory calibration diagnostics and require independent validation before operational use.

## 0. Methodological contract

- Analysis is read-only with respect to SQLite and source artifacts.
- Run IDs, random seed, dimensions, and output locations are centralized below.
- Company labels were generated without exposing deterministic score results to the OpenAI calibration prompt.
- Primary analysis uses `ai_enriched_v1`; the broad pilot is a negative/uncertainty stress test because it contains only one KEEP.
- `UNKNOWN` is analyzed separately and also grouped with non-KEEP only where explicitly stated.
- Page-score maxima, top-N summaries, repeated evidence, content-unique variants, and model-visible variants are compared rather than assuming one company aggregation.
- Shared domains, duplicate content, missing score coverage, and website size are treated as methodological risks.

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import sqlite3
from collections import Counter
from pathlib import Path
from urllib.parse import urlsplit

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from sklearn.metrics import average_precision_score, precision_recall_curve, roc_auc_score
from ai_ecosystem.calibration.grounding import ground_lexical_signal

RANDOM_SEED = 20260831
ANALYSIS_VERSION = "v1"
DIMENSIONS = ("firm_profile", "technology", "innovation", "ecosystem")
AI_DIMENSIONS = ("technology", "innovation", "ecosystem")
COMPONENTS = ("query_score", "keyword_score", "boost_score", "negative_score", "final_score")
LABEL_ORDER = ("KEEP", "UNKNOWN", "EXCLUDE")
LABEL_COLORS = {"KEEP": "#18794e", "UNKNOWN": "#b7791f", "EXCLUDE": "#b42318"}

RUNS = {
    "enriched": {
        "sample_id": "ai_enriched_v1",
        "calibration_run_id": "calibration_20260830T181238Z_3cf58aed",
        "scoring_run_id": "scoring_20260830T185247Z_839a147c",
        "package_run_id": "company_calibration_evidence_v1_ai_enriched_v1",
        "expected_companies": 500,
        "expected_scored_pages": 54951,
    },
    "pilot": {
        "sample_id": "pilot_v0_2",
        "calibration_run_id": "calibration_20260825T141041Z_ee153c3e",
        "scoring_run_id": "scoring_20260715T131248Z_f305ef18",
        "package_run_id": "company_calibration_evidence_v1_pilot_v0_2",
        "expected_companies": 149,
        "expected_scored_pages": 599,
    },
}


def find_project_root() -> Path:
    for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "src").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate project root")


PROJECT_ROOT = find_project_root()
DB_PATH = PROJECT_ROOT / "db" / "ai_ecosystem.sqlite"
ANALYSIS_ID = "company_calibration_analysis_v1_3cf58aed_839a147c"
OUTPUT_DIR = PROJECT_ROOT / "data" / "interim" / "calibration_analysis" / ANALYSIS_ID
CACHE_DIR = OUTPUT_DIR / "cache"
FIGURE_DIR = OUTPUT_DIR / "figures"
TABLE_DIR = OUTPUT_DIR / "tables"
for directory in (OUTPUT_DIR, CACHE_DIR, FIGURE_DIR, TABLE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

REBUILD_CACHE = os.environ.get("CALIBRATION_ANALYSIS_REBUILD", "0") == "1"
np.random.seed(RANDOM_SEED)
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 160)
print(f"Project root: {PROJECT_ROOT}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Forced cache rebuild: {REBUILD_CACHE}")

In [ ]:
def readonly_connection() -> sqlite3.Connection:
    connection = sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True)
    connection.execute("PRAGMA query_only = ON")
    return connection


def read_sql(query: str, params: tuple = ()) -> pd.DataFrame:
    with readonly_connection() as connection:
        return pd.read_sql_query(query, connection, params=params)


def calibration_dir(key: str) -> Path:
    return PROJECT_ROOT / "data/interim/calibration/openai" / RUNS[key]["calibration_run_id"]


def package_dir(key: str) -> Path:
    return PROJECT_ROOT / "data/interim/calibration/company_evidence" / RUNS[key]["package_run_id"]


def scoring_dir(key: str) -> Path:
    return PROJECT_ROOT / "data/interim/scoring" / RUNS[key]["scoring_run_id"]


def fingerprint(path: Path, include_hash: bool = True) -> dict:
    stat = path.stat()
    result = {"path": str(path.relative_to(PROJECT_ROOT)), "size": stat.st_size, "mtime_ns": stat.st_mtime_ns}
    if include_hash:
        digest = hashlib.sha256()
        with path.open("rb") as stream:
            for block in iter(lambda: stream.read(1024 * 1024), b""):
                digest.update(block)
        result["sha256"] = digest.hexdigest()
    return result


def save_figure(name: str) -> None:
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / f"{name}.png", dpi=180, bbox_inches="tight")
    plt.show()


def normalized_host(value: str | None) -> str:
    try:
        host = (urlsplit(value or "").hostname or "").casefold().rstrip(".")
    except ValueError:
        return ""
    return host[4:] if host.startswith("www.") else host


def related_host(url: str | None, domain: str | None) -> bool:
    left = normalized_host(url)
    right = normalized_host(f"https://{domain or ''}")
    return bool(left and right and (left == right or left.endswith(f".{right}") or right.endswith(f".{left}")))


def finite_feature(frame: pd.DataFrame, column: str) -> np.ndarray:
    values = pd.to_numeric(frame[column], errors="coerce").to_numpy(dtype=float)
    finite = values[np.isfinite(values)]
    floor = float(finite.min() - max(1.0, finite.std())) if len(finite) else -1.0
    return np.where(np.isfinite(values), values, floor)

## 1. Integrity gate, source runs, and reproducibility manifest

In [ ]:
required_paths = [DB_PATH, PROJECT_ROOT / "config/calibration_enrichment.yaml"]
for key in RUNS:
    required_paths += [
        calibration_dir(key) / "run_summary.json",
        calibration_dir(key) / "results.csv",
        calibration_dir(key) / "results.json",
        calibration_dir(key) / "lexical_signals.csv",
        package_dir(key) / "manifest.json",
        package_dir(key) / "package_summary.csv",
        scoring_dir(key) / "scoring_summary.json",
    ]
missing = [str(path) for path in required_paths if not path.exists()]
assert not missing, f"Missing required artifacts: {missing}"

calibration_runs = read_sql(
    "SELECT * FROM calibration_runs WHERE calibration_run_id IN (?, ?) ORDER BY calibration_run_id",
    tuple(RUNS[key]["calibration_run_id"] for key in ("pilot", "enriched")),
)
scoring_runs = read_sql(
    "SELECT * FROM scoring_runs WHERE scoring_run_id IN (?, ?) ORDER BY scoring_run_id",
    tuple(RUNS[key]["scoring_run_id"] for key in ("pilot", "enriched")),
)
assert set(calibration_runs.status) == {"completed"}
assert set(scoring_runs.status) == {"completed"}

score_contract = read_sql(
    """
    SELECT scoring_run_id, dimension, extraction_version, knowledge_base_id,
           knowledge_base_version, scoring_profile_id, scoring_profile_version,
           COUNT(*) AS row_count, COUNT(DISTINCT extracted_page_id) AS page_count
    FROM page_scores WHERE scoring_run_id IN (?, ?)
    GROUP BY scoring_run_id, dimension, extraction_version, knowledge_base_id,
             knowledge_base_version, scoring_profile_id, scoring_profile_version
    ORDER BY scoring_run_id, dimension
    """,
    tuple(RUNS[key]["scoring_run_id"] for key in ("pilot", "enriched")),
)
assert set(score_contract.dimension) == set(DIMENSIONS)
assert set(score_contract.extraction_version) == {"v1"}
assert set(score_contract.knowledge_base_version) == {"v1"}
assert score_contract.groupby("scoring_run_id").dimension.nunique().eq(4).all()

source_state = [fingerprint(path) for path in required_paths if path != DB_PATH]
manifest_path = OUTPUT_DIR / "analysis_manifest.json"
old_manifest = json.loads(manifest_path.read_text()) if manifest_path.exists() else {}
cache_valid = old_manifest.get("runs") == RUNS and old_manifest.get("sources") == source_state
manifest = {
    "analysis_id": ANALYSIS_ID,
    "analysis_version": ANALYSIS_VERSION,
    "random_seed": RANDOM_SEED,
    "runs": RUNS,
    "database": fingerprint(DB_PATH, include_hash=False),
    "sources": source_state,
}
manifest_path.write_text(json.dumps(manifest, indent=2, sort_keys=True) + "\n", encoding="utf-8")
print(f"Existing cache matches source manifest: {cache_valid}")
display(calibration_runs[["calibration_run_id", "sample_id", "status", "selected_count", "keep_count", "unknown_count", "exclude_count"]])
display(scoring_runs[["scoring_run_id", "sample_id", "status", "pages_scored", "pages_failed", "runtime_seconds"]])
display(score_contract)

## 2. Assemble company labels, sample metadata, and model-visible evidence

The analytical company unit is keyed by `(sample, company_id)`. Package summaries describe available evidence. Blind package JSON identifies which pages actually received content in the OpenAI prompt; metadata-only pages were not model-visible.

In [ ]:
def load_companies(key: str) -> pd.DataFrame:
    run = RUNS[key]
    results = pd.read_csv(calibration_dir(key) / "results.csv")
    packages = pd.read_csv(package_dir(key) / "package_summary.csv")
    frame = results.merge(
        packages,
        on=["company_id", "company_name", "domain", "shared_domain"],
        how="left",
        validate="one_to_one",
    )
    frame.insert(0, "sample", key)
    frame["sample_id"] = run["sample_id"]
    if key == "enriched":
        sample = read_sql(
            """SELECT company_id, sample_rank, nace_code AS sample_nace_code,
                      nace_description, enrichment_stratum, normalized_domain AS sample_domain
               FROM sample_companies WHERE sample_id=?""",
            (run["sample_id"],),
        )
        frame = frame.merge(sample, on="company_id", how="left", validate="one_to_one")
        assert frame.enrichment_stratum.notna().all()
    else:
        firms = read_sql(
            """SELECT f.bvd_id AS company_id, f.nace_code AS sample_nace_code
               FROM firms f JOIN calibration_results r ON r.company_id=f.bvd_id
               WHERE r.calibration_run_id=?""",
            (run["calibration_run_id"],),
        ).drop_duplicates("company_id")
        frame = frame.merge(firms, on="company_id", how="left", validate="one_to_one")
        frame["enrichment_stratum"] = "BROAD_PILOT"
        frame["nace_description"] = pd.NA
    for column in ("shared_domain", "explicit_ai", "ai_adjacent", "company_attributed_ai", "uncertainty_present"):
        frame[column] = frame[column].astype("boolean")
    assert len(frame) == run["expected_companies"]
    assert frame.company_id.is_unique
    return frame


companies = pd.concat([load_companies("pilot"), load_companies("enriched")], ignore_index=True, sort=False)
assert not set(companies.query("sample == 'pilot'").company_id) & set(companies.query("sample == 'enriched'").company_id)

package_records = []
visible_content = {}
package_page_catalogs = {}
for key in RUNS:
    blind_dir = package_dir(key) / "blind_inputs"
    for company_id in companies.loc[companies["sample"].eq(key), "company_id"]:
        payload = json.loads((blind_dir / f"{company_id}.json").read_text(encoding="utf-8"))
        pages = payload.get("pages", [])
        package_page_catalogs[(key, company_id)] = pages
        for page in pages:
            page_id = int(page["scraped_page_id"])
            status = page.get("content_status")
            visible = status in {"complete", "partially_truncated"}
            package_records.append({
                "sample": key,
                "company_id": company_id,
                "scraped_page_id": page_id,
                "extracted_page_id": int(page["extracted_page_id"]),
                "package_content_status": status,
                "visible_to_llm": visible,
                "package_url": page.get("url"),
            })
            if visible:
                visible_content[(key, company_id, page_id)] = str(page.get("content") or "")
package_pages = pd.DataFrame(package_records)
assert not package_pages.duplicated(["sample", "company_id", "scraped_page_id"]).any()

coverage = companies.groupby(["sample", "label"], observed=True).agg(
    companies=("company_id", "size"),
    usable=("usable_page_count", lambda x: int((x > 0).sum())),
    no_usable=("usable_page_count", lambda x: int((x == 0).sum())),
    shared=("shared_domain", "sum"),
    median_pages=("usable_page_count", "median"),
    max_pages=("usable_page_count", "max"),
    median_words=("total_words", "median"),
).reset_index()
display(coverage)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.countplot(data=companies, x="label", hue="sample", order=LABEL_ORDER, ax=axes[0])
axes[0].set_title("OpenAI labels by calibration sample")
sns.boxplot(data=companies, x="label", y="usable_page_count", order=LABEL_ORDER, showfliers=False, ax=axes[1])
axes[1].set_title("Usable pages by label")
axes[1].set_yscale("symlog")
sns.boxplot(data=companies, x="label", y="total_words", order=LABEL_ORDER, showfliers=False, ax=axes[2])
axes[2].set_title("Extracted words by label")
axes[2].set_yscale("symlog")
save_figure("01_sample_and_coverage_audit")

## 3. OpenAI lexical signals and historical grounding repair

OpenAI lexical signals are candidate observations, not ground truth. The historical evidence Markdown exposed both `scraped_page_id` and `extracted_page_id`, but the response schema requested only an ambiguous `page_id`. This audit therefore resolves every signal against both ID namespaces and the supplied URL. A unique URL is the strongest resolver; ID/URL conflicts and namespace collisions remain explicit provenance fields rather than being silently discarded.

Grounding is evaluated only against the resolved page content that was visible to the model. Strict matching is a case-insensitive literal substring check. Normalized matching applies Unicode NFKC normalization, case folding, punctuation/hyphen normalization, and whitespace collapsing. It does **not** use semantic similarity. Phrase and context results remain separate because the model generally copied phrases more reliably than its often paraphrased or translated `short_context`.

For future calibration schemas, expose one explicit stable page identifier, preferably `scraped_page_id`, and request a clearly named verbatim context quote. This notebook repairs the historical run only; it does not alter model outputs or labels.

In [ ]:
lexical_frames = []
for key in RUNS:
    frame = pd.read_csv(calibration_dir(key) / "lexical_signals.csv")
    frame.insert(0, "sample", key)
    frame["page_id"] = frame["page_id"].astype("string")
    frame["phrase_verified"] = frame.phrase_verified.astype("boolean")
    frame["context_verified"] = frame.context_verified.astype("boolean")
    lexical_frames.append(frame)
lexical = pd.concat(lexical_frames, ignore_index=True)

grounding_records = []
for signal_record in lexical.to_dict(orient="records"):
    pages = package_page_catalogs[(signal_record["sample"], signal_record["company_id"])]
    grounding_records.append(ground_lexical_signal(signal_record, pages))
lexical = pd.concat([lexical.reset_index(drop=True), pd.DataFrame(grounding_records)], axis=1)
lexical.to_parquet(CACHE_DIR / "lexical_signals.parquet", index=False)

grounding = lexical.groupby(["sample", "signal_type"], observed=True).agg(
    signals=("phrase", "size"),
    resolved=("resolved_scraped_page_id", "count"),
    unresolved=("page_resolution_method", lambda values: int(values.eq("unresolved").sum())),
    id_url_conflicts=("page_id_url_conflict", "sum"),
    visible_pages=("page_visible_to_model", "sum"),
    strict_phrase_grounded=("phrase_grounded_strict", "sum"),
    normalized_phrase_grounded=("phrase_grounded_normalized", "sum"),
    strict_context_grounded=("context_grounded_strict", "sum"),
    normalized_context_grounded=("context_grounded_normalized", "sum"),
).reset_index()
for column in ("strict_phrase_grounded", "normalized_phrase_grounded", "strict_context_grounded", "normalized_context_grounded"):
    grounding[f"{column}_rate"] = grounding[column] / grounding.signals

grounding_by_label = lexical.groupby(["sample", "company_label"], observed=True).agg(
    signals=("phrase", "size"),
    resolved=("resolved_scraped_page_id", "count"),
    strict_phrase_grounded=("phrase_grounded_strict", "sum"),
    normalized_phrase_grounded=("phrase_grounded_normalized", "sum"),
    strict_context_grounded=("context_grounded_strict", "sum"),
    normalized_context_grounded=("context_grounded_normalized", "sum"),
).reset_index()
for column in ("strict_phrase_grounded", "normalized_phrase_grounded", "strict_context_grounded", "normalized_context_grounded"):
    grounding_by_label[f"{column}_rate"] = grounding_by_label[column] / grounding_by_label.signals

resolution_summary = lexical.groupby(
    ["page_resolution_source", "page_resolution_method", "page_id_namespace"], observed=True, dropna=False
).agg(
    signals=("phrase", "size"),
    conflicts=("page_id_url_conflict", "sum"),
    strict_phrase_grounded=("phrase_grounded_strict", "sum"),
    normalized_phrase_grounded=("phrase_grounded_normalized", "sum"),
).reset_index()

conflicts = lexical.loc[lexical.page_id_url_conflict].copy()
conflict_summary = pd.DataFrame([{
    "conflicts": len(conflicts),
    "id_selected_phrase_strict": int(conflicts.id_selected_phrase_grounded_strict.sum()),
    "id_selected_phrase_normalized": int(conflicts.id_selected_phrase_grounded_normalized.sum()),
    "url_selected_phrase_strict": int(conflicts.url_selected_phrase_grounded_strict.sum()),
    "url_selected_phrase_normalized": int(conflicts.url_selected_phrase_grounded_normalized.sum()),
}])

grounding.to_csv(TABLE_DIR / "lexical_grounding_by_category.csv", index=False)
grounding_by_label.to_csv(TABLE_DIR / "lexical_grounding_by_company_label.csv", index=False)
resolution_summary.to_csv(TABLE_DIR / "lexical_page_resolution_summary.csv", index=False)
conflicts[[
    "sample", "company_id", "company_label", "signal_type", "page_id", "url", "phrase",
    "resolved_scraped_page_id", "resolved_extracted_page_id", "page_resolution_method",
    "page_id_namespace", "id_selected_scraped_page_id", "url_selected_scraped_page_id",
    "id_selected_phrase_grounded_strict", "url_selected_phrase_grounded_strict",
]].to_csv(TABLE_DIR / "lexical_id_url_conflicts.csv", index=False)

display(grounding)
display(grounding_by_label)
display(resolution_summary)
display(conflict_summary)

signal_counts = lexical.groupby(["sample", "company_label", "signal_type"], observed=True).size().rename("count").reset_index()
fig, axes = plt.subplots(1, 2, figsize=(19, 7))
sns.barplot(
    data=signal_counts.query("sample == 'enriched'"), x="count", y="signal_type",
    hue="company_label", hue_order=LABEL_ORDER, ax=axes[0],
)
axes[0].set_title("Enriched lexical-signal suggestions")

rate_plot = grounding.melt(
    id_vars=["sample", "signal_type"],
    value_vars=[
        "strict_phrase_grounded_rate", "normalized_phrase_grounded_rate",
        "strict_context_grounded_rate", "normalized_context_grounded_rate",
    ],
    var_name="grounding_measure", value_name="rate",
)
sns.barplot(
    data=rate_plot.query("sample == 'enriched'"), x="rate", y="signal_type",
    hue="grounding_measure", ax=axes[1],
)
axes[1].set_title("Enriched page-local grounding rates")
axes[1].set_xlim(0, 1)
save_figure("02_lexical_signal_grounding")

## 4. Load coherent page-score contexts

Each scoring run is pivoted to one row per extracted page. No score rows are mixed across scoring runs. The page unit retains all four dimensions and components, content hashes for duplicate detection, and structural metadata. Company attachment uses the calibration domain and is explicitly audited for cross-host and shared-domain risk.

In [ ]:
def load_page_scores(key: str) -> pd.DataFrame:
    cache_path = CACHE_DIR / f"page_scores_{key}.parquet"
    if cache_path.exists() and cache_valid and not REBUILD_CACHE:
        return pd.read_parquet(cache_path)
    long = read_sql(
        """
        SELECT ps.scoring_run_id, ps.extracted_page_id, ps.scraped_page_id, ps.domain, ps.url,
               ps.dimension, ps.query_score, ps.keyword_score, ps.boost_score,
               ps.negative_score, ps.final_score, ep.content_sha256, ep.extraction_version,
               pf.title, pf.word_count, pf.page_type_guess, pf.language, sp.final_url
        FROM page_scores ps
        JOIN extracted_pages ep ON ep.extracted_page_id=ps.extracted_page_id
        JOIN scraped_pages sp ON sp.scraped_page_id=ps.scraped_page_id
        LEFT JOIN page_features pf ON pf.extracted_page_id=ps.extracted_page_id
        WHERE ps.scoring_run_id=? ORDER BY ps.scraped_page_id, ps.dimension
        """,
        (RUNS[key]["scoring_run_id"],),
    )
    assert not long.duplicated(["extracted_page_id", "dimension"]).any()
    identity = ["scoring_run_id", "extracted_page_id", "scraped_page_id", "domain", "url"]
    metadata_columns = ["content_sha256", "extraction_version", "title", "word_count", "page_type_guess", "language", "final_url"]
    metadata = long[identity + metadata_columns].drop_duplicates(identity)
    wide = long.pivot(index=identity, columns="dimension", values=list(COMPONENTS)).reset_index()
    wide.columns = [
        f"{column[1]}__{column[0]}" if isinstance(column, tuple) and column[1] else column[0]
        for column in wide.columns
    ]
    wide = wide.merge(metadata, on=identity, how="left", validate="one_to_one")
    wide.insert(0, "sample", key)
    wide["url_host_matches_stored_domain"] = [related_host(url, domain) for url, domain in zip(wide.url, wide.domain)]
    wide["duplicate_group_size"] = wide.groupby("content_sha256", dropna=False).scraped_page_id.transform("size")
    wide.to_parquet(cache_path, index=False)
    return wide


page_scores = pd.concat([load_page_scores("pilot"), load_page_scores("enriched")], ignore_index=True, sort=False)
assert page_scores.groupby("sample").size().to_dict() == {key: value["expected_scored_pages"] for key, value in RUNS.items()}

company_keys = companies[["sample", "company_id", "company_name", "domain", "label", "confidence", "shared_domain", "enrichment_stratum", "sample_nace_code", "nace_description"]]
company_pages = company_keys.merge(page_scores, on=["sample", "domain"], how="left", validate="many_to_many")
company_pages = company_pages.merge(
    package_pages[["sample", "company_id", "scraped_page_id", "package_content_status", "visible_to_llm"]],
    on=["sample", "company_id", "scraped_page_id"], how="left", validate="many_to_one",
)
company_pages["visible_to_llm"] = company_pages.visible_to_llm.fillna(False).astype(bool)
company_pages["has_score"] = company_pages.scraped_page_id.notna()
company_pages["content_identity"] = company_pages.content_sha256.where(
    company_pages.content_sha256.notna(), "page:" + company_pages.scraped_page_id.astype("Int64").astype(str)
)
company_pages["content_unique_rank"] = company_pages.groupby(
    ["sample", "company_id", "content_identity"], dropna=False
).scraped_page_id.rank(method="first")
company_pages["content_unique"] = company_pages.content_unique_rank.eq(1) & company_pages.has_score

scope_audit = company_pages.groupby(["sample", "company_id"], observed=True).agg(
    scored_pages=("has_score", "sum"),
    visible_scored_pages=("visible_to_llm", "sum"),
    unique_content_pages=("content_unique", "sum"),
    cross_host_pages=("url_host_matches_stored_domain", lambda x: int((~x.fillna(True)).sum())),
).reset_index()
companies = companies.merge(scope_audit, on=["sample", "company_id"], how="left", validate="one_to_one")
for column in ("scored_pages", "visible_scored_pages", "unique_content_pages", "cross_host_pages"):
    companies[column] = companies[column].fillna(0).astype(int)
display(companies.groupby(["sample", "label"], observed=True)[["scored_pages", "visible_scored_pages", "cross_host_pages"]].agg(["count", "median", "max"]))

## 5. Candidate company-level score features

Features are computed independently for four evidence scopes:

1. all scored pages, raw;
2. all scored pages, one row per content hash;
3. only pages visible to the OpenAI calibration model, raw;
4. model-visible pages, one row per content hash.

For every dimension the notebook calculates maxima, second/third order statistics, top-3/5/10 means, medians, upper percentiles, score-sign counts, maxima gaps, component maxima, and counts/shares above empirical enriched-page quantiles. The quantiles describe this calibration score distribution; they are not production thresholds.

In [ ]:
enriched_page_scores = page_scores.loc[page_scores["sample"].eq("enriched")]
page_thresholds = {
    dimension: {q: float(enriched_page_scores[f"{dimension}__final_score"].quantile(q)) for q in (0.75, 0.90, 0.95, 0.99)}
    for dimension in DIMENSIONS
}
display(pd.DataFrame(page_thresholds).rename_axis("page quantile"))


def top_mean(values: np.ndarray, n: int) -> float:
    ordered = np.sort(values[np.isfinite(values)])[::-1]
    return float(ordered[:n].mean()) if len(ordered) else np.nan


def aggregate_scope(group: pd.DataFrame, prefix: str) -> dict:
    output = {f"{prefix}__page_count": int(group.scraped_page_id.notna().sum())}
    for dimension in DIMENSIONS:
        final = group[f"{dimension}__final_score"].dropna().to_numpy(dtype=float)
        ordered = np.sort(final)[::-1]
        stats_map = {
            "max": ordered[0] if len(ordered) else np.nan,
            "rank2": ordered[1] if len(ordered) > 1 else np.nan,
            "rank3": ordered[2] if len(ordered) > 2 else np.nan,
            "top3_mean": top_mean(final, 3),
            "top5_mean": top_mean(final, 5),
            "top10_mean": top_mean(final, 10),
            "mean": float(final.mean()) if len(final) else np.nan,
            "median": float(np.median(final)) if len(final) else np.nan,
            "p75": float(np.quantile(final, 0.75)) if len(final) else np.nan,
            "p90": float(np.quantile(final, 0.90)) if len(final) else np.nan,
            "p95": float(np.quantile(final, 0.95)) if len(final) else np.nan,
            "positive_count": int((final > 0).sum()),
            "positive_share": float((final > 0).mean()) if len(final) else np.nan,
            "max_rank2_gap": float(ordered[0] - ordered[1]) if len(ordered) > 1 else np.nan,
            "max_top3_gap": float(ordered[0] - top_mean(final, 3)) if len(final) else np.nan,
        }
        for name, value in stats_map.items():
            output[f"{prefix}__{dimension}__final__{name}"] = value
        for quantile, threshold in page_thresholds[dimension].items():
            count = int((final >= threshold).sum())
            suffix = int(quantile * 100)
            output[f"{prefix}__{dimension}__final__count_ge_q{suffix}"] = count
            output[f"{prefix}__{dimension}__final__share_ge_q{suffix}"] = count / len(final) if len(final) else np.nan
        for component in COMPONENTS[:-1]:
            values = group[f"{dimension}__{component}"].dropna().to_numpy(dtype=float)
            short = component.removesuffix("_score")
            output[f"{prefix}__{dimension}__{short}__max"] = float(values.max()) if len(values) else np.nan
            output[f"{prefix}__{dimension}__{short}__top3_mean"] = top_mean(values, 3)
    return output


feature_rows = []
for (sample, company_id), raw_group in company_pages.groupby(["sample", "company_id"], sort=True, observed=True):
    scored = raw_group.loc[raw_group.has_score]
    scopes = {
        "all_raw": scored,
        "all_unique": scored.loc[scored.content_unique],
        "visible_raw": scored.loc[scored.visible_to_llm],
        "visible_unique": scored.loc[scored.visible_to_llm & scored.content_unique],
    }
    row = {"sample": sample, "company_id": company_id}
    for prefix, frame in scopes.items():
        row.update(aggregate_scope(frame, prefix))
    feature_rows.append(row)

company_features = companies.merge(pd.DataFrame(feature_rows), on=["sample", "company_id"], how="left", validate="one_to_one")
company_features["duplicate_page_share"] = 1 - company_features.unique_content_pages.div(company_features.scored_pages.replace(0, np.nan))
company_features["cross_host_page_share"] = company_features.cross_host_pages.div(company_features.scored_pages.replace(0, np.nan))
for scope in ("all_raw", "all_unique", "visible_raw", "visible_unique"):
    for statistic in ("max", "top3_mean", "top5_mean"):
        columns = [f"{scope}__{dimension}__final__{statistic}" for dimension in AI_DIMENSIONS]
        company_features[f"{scope}__ai_dimensions__final__{statistic}_max"] = company_features[columns].max(axis=1)
        company_features[f"{scope}__technology_innovation__final__{statistic}_mean"] = company_features[columns[:2]].mean(axis=1)
    repeat_columns = [f"{scope}__{dimension}__final__count_ge_q95" for dimension in AI_DIMENSIONS]
    company_features[f"{scope}__ai_dimensions__repeated_q95_pages"] = company_features[repeat_columns].sum(axis=1)

company_features.to_parquet(CACHE_DIR / "company_features.parquet", index=False)
page_scores.to_parquet(CACHE_DIR / "page_scores_all.parquet", index=False)
assert len(company_features) == 649
enriched = company_features.query("sample == 'enriched'").copy()
pilot = company_features.query("sample == 'pilot'").copy()
display(company_features.groupby(["sample", "label"], observed=True).size().rename("companies"))

## 6. Score distributions, class overlap, and dimension relationships

In [ ]:
primary_features = [f"all_unique__{dimension}__final__top3_mean" for dimension in DIMENSIONS]
long_scores = enriched.melt(
    id_vars=["company_id", "label"], value_vars=primary_features,
    var_name="dimension", value_name="top3_mean",
)
long_scores["dimension"] = long_scores.dimension.str.extract(r"all_unique__(.*?)__final")[0]

fig, axes = plt.subplots(2, 2, figsize=(17, 12))
for dimension, ax in zip(DIMENSIONS, axes.flat):
    sns.boxplot(
        data=long_scores.query("dimension == @dimension"), x="label", y="top3_mean",
        order=LABEL_ORDER, palette=LABEL_COLORS, hue="label", legend=False, showfliers=False, ax=ax,
    )
    sns.stripplot(data=long_scores.query("dimension == @dimension"), x="label", y="top3_mean", order=LABEL_ORDER, color="black", alpha=.25, size=2, ax=ax)
    ax.set_title(f"{dimension}: content-unique top-3 mean")
save_figure("03_score_boxplots_by_label")

fig, axes = plt.subplots(2, 2, figsize=(17, 12))
for dimension, ax in zip(DIMENSIONS, axes.flat):
    column = f"all_unique__{dimension}__final__top3_mean"
    for label in LABEL_ORDER:
        values = enriched.loc[enriched.label.eq(label), column].dropna().sort_values().to_numpy()
        if len(values):
            ax.step(values, np.arange(1, len(values) + 1) / len(values), where="post", label=label, color=LABEL_COLORS[label])
    ax.set_title(f"ECDF: {dimension} top-3 mean")
    ax.set_xlabel("score")
    ax.set_ylabel("cumulative share")
    ax.legend()
save_figure("04_score_ecdfs_by_label")

correlation_features = primary_features + [
    "scored_pages", "total_words", "all_unique__ai_dimensions__repeated_q95_pages"
]
fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(enriched[correlation_features].corr(method="spearman"), cmap="vlag", center=0, annot=True, fmt=".2f", ax=ax)
ax.set_title("Spearman relationships among company evidence summaries")
save_figure("05_feature_correlation_heatmap")

effect_rows = []
for feature in primary_features:
    groups = {label: enriched.loc[enriched.label.eq(label), feature].dropna() for label in LABEL_ORDER}
    for left, right in (("KEEP", "EXCLUDE"), ("KEEP", "UNKNOWN"), ("UNKNOWN", "EXCLUDE")):
        if len(groups[left]) and len(groups[right]):
            statistic, pvalue = stats.mannwhitneyu(groups[left], groups[right], alternative="two-sided")
            effect_rows.append({
                "feature": feature, "comparison": f"{left}_vs_{right}", "n_left": len(groups[left]),
                "n_right": len(groups[right]), "mann_whitney_u": statistic, "pvalue_descriptive": pvalue,
                "rank_biserial": 2 * statistic / (len(groups[left]) * len(groups[right])) - 1,
            })
effect_sizes = pd.DataFrame(effect_rows)
effect_sizes.to_csv(TABLE_DIR / "descriptive_effect_sizes.csv", index=False)
display(effect_sizes)

## 7. Website-size bias and one-page-spike diagnostics

Maxima can increase mechanically as a website supplies more pages. The plots compare maximum and repeated-evidence summaries against page count. These relationships are descriptive and do not establish causal bias.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(17, 12))
for dimension, ax in zip(DIMENSIONS, axes.flat):
    sns.scatterplot(
        data=enriched, x="scored_pages", y=f"all_unique__{dimension}__final__max",
        hue="label", hue_order=LABEL_ORDER, palette=LABEL_COLORS, alpha=.7, ax=ax,
    )
    ax.set_xscale("symlog")
    ax.set_title(f"{dimension}: maximum versus pages")
save_figure("06_website_size_vs_maximum")

fig, axes = plt.subplots(1, 3, figsize=(19, 6))
for dimension, ax in zip(AI_DIMENSIONS, axes):
    sns.scatterplot(
        data=enriched, x=f"all_unique__{dimension}__final__max",
        y=f"all_unique__{dimension}__final__top3_mean", hue="label",
        hue_order=LABEL_ORDER, palette=LABEL_COLORS, alpha=.7, ax=ax,
    )
    ax.plot(ax.get_xlim(), ax.get_xlim(), linestyle="--", color="grey", linewidth=1)
    ax.set_title(f"{dimension}: spike vs repeated evidence")
save_figure("07_maximum_vs_top3_mean")

size_correlations = []
for label in ("ALL", *LABEL_ORDER):
    frame = enriched if label == "ALL" else enriched.loc[enriched.label.eq(label)]
    for feature in primary_features:
        valid = frame[["scored_pages", feature]].dropna()
        rho, pvalue = stats.spearmanr(valid.scored_pages, valid[feature]) if len(valid) > 2 else (np.nan, np.nan)
        size_correlations.append({"label": label, "feature": feature, "n": len(valid), "spearman_rho": rho, "pvalue_descriptive": pvalue})
size_correlations = pd.DataFrame(size_correlations)
size_correlations.to_csv(TABLE_DIR / "website_size_correlations.csv", index=False)
display(size_correlations)

## 8. High-scoring non-KEEPs and low-scoring KEEPs: page-level casebooks

In [ ]:
def top_company_pages(company_ids, dimension: str, limit_per_company: int = 3) -> pd.DataFrame:
    subset = company_pages.loc[
        company_pages.company_id.isin(company_ids) & company_pages.has_score,
        ["sample", "company_id", "company_name", "domain", "label", "scraped_page_id", "url", "title",
         "visible_to_llm", "content_unique", f"{dimension}__query_score", f"{dimension}__keyword_score",
         f"{dimension}__boost_score", f"{dimension}__negative_score", f"{dimension}__final_score"],
    ].copy()
    return (
        subset.sort_values(["company_id", f"{dimension}__final_score", "scraped_page_id"], ascending=[True, False, True])
        .groupby("company_id", observed=True).head(limit_per_company)
    )


casebook_index = []
for dimension in AI_DIMENSIONS:
    feature = f"all_unique__{dimension}__final__top3_mean"
    high_nonkeep = enriched.loc[enriched.label.ne("KEEP") & enriched.scored_pages.gt(0)].nlargest(30, feature)
    low_keep = enriched.loc[enriched.label.eq("KEEP") & enriched.scored_pages.gt(0)].nsmallest(30, feature)
    for case_type, frame in (("high_nonkeep", high_nonkeep), ("low_keep", low_keep)):
        company_path = TABLE_DIR / f"{dimension}_{case_type}_companies.csv"
        page_path = TABLE_DIR / f"{dimension}_{case_type}_pages.csv"
        frame[["company_id", "company_name", "domain", "label", "confidence", "shared_domain", "enrichment_stratum", "scored_pages", feature]].to_csv(company_path, index=False)
        top_company_pages(frame.company_id, dimension).to_csv(page_path, index=False)
        casebook_index.append({"dimension": dimension, "case_type": case_type, "companies": len(frame), "company_table": company_path.name, "page_table": page_path.name})
casebook_index = pd.DataFrame(casebook_index)
display(casebook_index)

strict_grounded_lexical = lexical.loc[lexical.phrase_grounded_strict & lexical["sample"].eq("enriched")].copy()
grounded_lexical = lexical.loc[lexical.phrase_grounded_normalized & lexical["sample"].eq("enriched")].copy()
grounded_lexical.groupby(["company_label", "signal_type", "normalized_phrase"], observed=True).size().rename("occurrences").reset_index().sort_values(
    ["company_label", "signal_type", "occurrences"], ascending=[True, True, False]
).to_csv(TABLE_DIR / "grounded_lexical_phrase_inventory.csv", index=False)

false_positive_phrases = grounded_lexical.loc[
    grounded_lexical.company_label.ne("KEEP") & grounded_lexical.signal_type.isin(["false_positive", "third_party_ai", "generic_marketing"])
].groupby(["signal_type", "normalized_phrase"], observed=True).agg(
    occurrences=("company_id", "size"), companies=("company_id", "nunique")
).reset_index().sort_values(["companies", "occurrences"], ascending=False)

missed_keep_phrases = grounded_lexical.loc[
    grounded_lexical.company_label.eq("KEEP") & grounded_lexical.signal_type.isin(["positive_ai", "ai_adjacent", "useful_context_pattern"])
].groupby(["signal_type", "normalized_phrase"], observed=True).agg(
    occurrences=("company_id", "size"), companies=("company_id", "nunique")
).reset_index().sort_values(["companies", "occurrences"], ascending=False)
false_positive_phrases.to_csv(TABLE_DIR / "candidate_downweighting_phrases.csv", index=False)
missed_keep_phrases.to_csv(TABLE_DIR / "candidate_vocabulary_phrases.csv", index=False)
display(false_positive_phrases.head(30))
display(missed_keep_phrases.head(30))

## 9. Candidate feature diagnostics and resource-constrained selection budgets

The target is KEEP versus all other labels for the primary resource-allocation view. KEEP versus EXCLUDE is reported separately to avoid silently treating UNKNOWN as a clean negative. Average precision and precision-recall curves are primary; ROC AUC is secondary. Budget tables show how many KEEPs and non-KEEPs would reach a costly downstream stage.

In [ ]:
candidate_features = [
    *[f"all_unique__{dimension}__final__{stat}" for dimension in AI_DIMENSIONS for stat in ("max", "top3_mean", "top5_mean", "p90")],
    "all_unique__ai_dimensions__final__max_max",
    "all_unique__ai_dimensions__final__top3_mean_max",
    "all_unique__technology_innovation__final__top3_mean_mean",
    "all_unique__ai_dimensions__repeated_q95_pages",
    "visible_unique__ai_dimensions__final__top3_mean_max",
]

metric_rows = []
for formulation, mask in {
    "KEEP_vs_all": enriched.scored_pages.gt(0),
    "KEEP_vs_EXCLUDE": enriched.scored_pages.gt(0) & enriched.label.isin(["KEEP", "EXCLUDE"]),
}.items():
    frame = enriched.loc[mask]
    y = frame.label.eq("KEEP").astype(int).to_numpy()
    for feature in candidate_features:
        scores = finite_feature(frame, feature)
        metric_rows.append({
            "formulation": formulation, "feature": feature, "companies": len(frame), "keeps": int(y.sum()),
            "baseline_keep_share": float(y.mean()), "average_precision": average_precision_score(y, scores),
            "roc_auc_secondary": roc_auc_score(y, scores),
        })
feature_metrics = pd.DataFrame(metric_rows).sort_values(["formulation", "average_precision"], ascending=[True, False])
feature_metrics.to_csv(TABLE_DIR / "candidate_feature_metrics.csv", index=False)
display(feature_metrics.groupby("formulation", observed=True).head(12))

top_features = feature_metrics.query("formulation == 'KEEP_vs_all'").head(6).feature.tolist()
fig, axes = plt.subplots(1, 2, figsize=(17, 7))
covered = enriched.loc[enriched.scored_pages.gt(0)]
y = covered.label.eq("KEEP").astype(int).to_numpy()
for feature in top_features:
    precision, recall, _ = precision_recall_curve(y, finite_feature(covered, feature))
    axes[0].plot(recall, precision, label=feature.replace("all_unique__", ""))
axes[0].axhline(y.mean(), linestyle="--", color="grey", label="KEEP baseline")
axes[0].set(xlabel="KEEP recall", ylabel="KEEP precision", title="Precision-recall: KEEP versus all")
axes[0].legend(fontsize=7)

budget_rows = []
for feature in candidate_features:
    ranked = covered.assign(_score=finite_feature(covered, feature)).sort_values(["_score", "company_id"], ascending=[False, True])
    for budget in (25, 50, 100, 125, 250):
        selected = ranked.head(min(budget, len(ranked)))
        counts = selected.label.value_counts()
        keep = int(counts.get("KEEP", 0))
        budget_rows.append({
            "feature": feature, "budget": budget, "selected": len(selected), "keep": keep,
            "unknown": int(counts.get("UNKNOWN", 0)), "exclude": int(counts.get("EXCLUDE", 0)),
            "keep_precision": keep / len(selected), "keep_recall": keep / max(1, int(y.sum())),
            "lift_over_covered_baseline": (keep / len(selected)) / y.mean(),
        })
budget_table = pd.DataFrame(budget_rows)
budget_table.to_csv(TABLE_DIR / "company_selection_budget_table.csv", index=False)
for feature in top_features:
    view = budget_table.loc[budget_table.feature.eq(feature)]
    axes[1].plot(view.budget, view.keep_recall, marker="o", label=feature.replace("all_unique__", ""))
axes[1].set(xlabel="companies sent downstream", ylabel="fraction of covered KEEPs retained", title="KEEP capture by fixed selection budget")
axes[1].legend(fontsize=7)
save_figure("08_precision_recall_and_budget_capture")

threshold_rows = []
for feature in top_features:
    for quantile in (0.50, 0.75, 0.90, 0.95):
        threshold = covered[feature].quantile(quantile)
        selected = covered.loc[covered[feature].ge(threshold)]
        counts = selected.label.value_counts()
        threshold_rows.append({
            "feature": feature, "calibration_quantile": quantile, "threshold": threshold,
            "selected": len(selected), "keep": int(counts.get("KEEP", 0)),
            "unknown": int(counts.get("UNKNOWN", 0)), "exclude": int(counts.get("EXCLUDE", 0)),
            "keep_precision": selected.label.eq("KEEP").mean(),
            "keep_recall": selected.label.eq("KEEP").sum() / max(1, int(y.sum())),
        })
threshold_table = pd.DataFrame(threshold_rows)
threshold_table.to_csv(TABLE_DIR / "exploratory_threshold_table.csv", index=False)
display(budget_table.loc[budget_table.feature.isin(top_features)].sort_values(["budget", "keep_precision"], ascending=[True, False]).groupby("budget").head(5))

## 10. ÖNACE enrichment strata and shared-domain attribution

In [ ]:
stratum_counts = enriched.groupby(["enrichment_stratum", "label"], observed=True).size().unstack(fill_value=0).reindex(columns=LABEL_ORDER)
shared_counts = enriched.groupby(["shared_domain", "label"], observed=True).size().unstack(fill_value=0).reindex(columns=LABEL_ORDER)
fig, axes = plt.subplots(1, 2, figsize=(17, 6))
stratum_counts.div(stratum_counts.sum(axis=1), axis=0).plot(kind="bar", stacked=True, color=[LABEL_COLORS[x] for x in LABEL_ORDER], ax=axes[0])
axes[0].set(title="Label composition within enrichment strata", ylabel="share")
axes[0].tick_params(axis="x", rotation=20)
shared_counts.div(shared_counts.sum(axis=1), axis=0).plot(kind="bar", stacked=True, color=[LABEL_COLORS[x] for x in LABEL_ORDER], ax=axes[1])
axes[1].set(title="Label composition by shared-domain flag", ylabel="share")
axes[1].set_xticklabels(["independent", "shared"], rotation=0)
save_figure("09_stratum_and_shared_domain_labels")

stratum_scores = enriched.melt(
    id_vars=["enrichment_stratum", "label"],
    value_vars=[f"all_unique__{dimension}__final__top3_mean" for dimension in AI_DIMENSIONS],
    var_name="dimension", value_name="score",
)
stratum_scores["dimension"] = stratum_scores.dimension.str.extract(r"all_unique__(.*?)__final")[0]
fig, axes = plt.subplots(1, 3, figsize=(19, 6))
for dimension, ax in zip(AI_DIMENSIONS, axes):
    sns.boxplot(data=stratum_scores.query("dimension == @dimension"), x="enrichment_stratum", y="score", hue="label", hue_order=LABEL_ORDER, palette=LABEL_COLORS, showfliers=False, ax=ax)
    ax.set_title(f"{dimension} within strata")
    ax.tick_params(axis="x", rotation=25)
save_figure("10_scores_within_enrichment_strata")

shared_casebook = enriched.loc[enriched.shared_domain.fillna(False)].sort_values(
    "all_unique__ai_dimensions__final__top3_mean_max", ascending=False
)[["company_id", "company_name", "domain", "label", "confidence", "enrichment_stratum", "mapped_company_count", "scored_pages", "cross_host_page_share", "duplicate_page_share", "all_unique__technology__final__top3_mean", "all_unique__innovation__final__top3_mean", "all_unique__ecosystem__final__top3_mean"]]
shared_casebook.to_csv(TABLE_DIR / "shared_domain_casebook.csv", index=False)
display(shared_casebook.head(30))

## 11. Cross-sample stress test

Thresholds are defined only from the enriched sample and then applied unchanged to score-covered broad-pilot companies. This cannot validate positive recall because the broad pilot has one KEEP and its scoring run covers only a subset of companies; it can expose negative/uncertainty leakage and enrichment dependence.

In [ ]:
cross_rows = []
for feature in top_features:
    reference = enriched.loc[enriched.scored_pages.gt(0), feature].dropna()
    for quantile in (0.75, 0.90, 0.95):
        threshold = reference.quantile(quantile)
        for sample_name, frame in (("enriched", enriched), ("pilot", pilot)):
            covered_frame = frame.loc[frame.scored_pages.gt(0)]
            selected = covered_frame.loc[covered_frame[feature].fillna(-np.inf).ge(threshold)]
            counts = selected.label.value_counts()
            cross_rows.append({
                "feature": feature, "enriched_reference_quantile": quantile, "threshold": threshold,
                "sample": sample_name, "score_covered_companies": len(covered_frame), "selected": len(selected),
                "keep": int(counts.get("KEEP", 0)), "unknown": int(counts.get("UNKNOWN", 0)),
                "exclude": int(counts.get("EXCLUDE", 0)),
                "selection_share": len(selected) / len(covered_frame) if len(covered_frame) else np.nan,
            })
cross_sample = pd.DataFrame(cross_rows)
cross_sample.to_csv(TABLE_DIR / "cross_sample_threshold_stress_test.csv", index=False)
display(cross_sample.query("enriched_reference_quantile == 0.90").sort_values(["feature", "sample"]))

fig, ax = plt.subplots(figsize=(13, 7))
plot_data = cross_sample.query("enriched_reference_quantile == 0.90").copy()
plot_data["feature_short"] = plot_data.feature.str.replace("all_unique__", "", regex=False).str.replace("__final__", ": ", regex=False)
sns.barplot(data=plot_data, x="selection_share", y="feature_short", hue="sample", ax=ax)
ax.set_title("Enriched q90 thresholds applied unchanged to each sample")
save_figure("11_cross_sample_stress_test")

## 12. Robustness checks and domain-cluster uncertainty

In [ ]:
conditions = {
    "all_companies": np.ones(len(enriched), dtype=bool),
    "score_covered": enriched.scored_pages.gt(0).to_numpy(),
    "independent_domains": (~enriched.shared_domain.fillna(False)).to_numpy(),
    "no_cross_host_pages": enriched.cross_host_pages.eq(0).to_numpy(),
    "at_least_3_pages": enriched.scored_pages.ge(3).to_numpy(),
    "KEEP_vs_EXCLUDE": enriched.label.isin(["KEEP", "EXCLUDE"]).to_numpy(),
}
robust_rows = []
for condition, mask in conditions.items():
    frame = enriched.loc[mask]
    y = frame.label.eq("KEEP").astype(int).to_numpy()
    if len(np.unique(y)) < 2:
        continue
    for feature in candidate_features:
        robust_rows.append({
            "condition": condition, "feature": feature, "companies": len(frame), "keeps": int(y.sum()),
            "average_precision": average_precision_score(y, finite_feature(frame, feature)),
        })
robustness = pd.DataFrame(robust_rows)
robustness.to_csv(TABLE_DIR / "robustness_metrics.csv", index=False)
heatmap = robustness.pivot(index="feature", columns="condition", values="average_precision")
fig, ax = plt.subplots(figsize=(13, 9))
sns.heatmap(heatmap, annot=True, fmt=".2f", cmap="viridis", ax=ax)
ax.set_title("Average precision under alternative evidence restrictions")
save_figure("12_robustness_heatmap")


def domain_cluster_bootstrap(frame: pd.DataFrame, feature: str, budget: int, iterations: int = 200) -> dict:
    rng = np.random.default_rng(RANDOM_SEED)
    domain_values = frame["domain"].fillna("__missing_domain__").astype(str).to_numpy()
    domains = np.unique(domain_values)
    group_indices = [np.flatnonzero(domain_values == domain) for domain in domains]
    estimates = []
    for _ in range(iterations):
        sampled_groups = rng.integers(0, len(group_indices), len(group_indices))
        row_indices = np.concatenate([group_indices[index] for index in sampled_groups])
        sample = frame.iloc[row_indices]
        ranked = sample.assign(_score=finite_feature(sample, feature)).sort_values(
            ["_score", "company_id"], ascending=[False, True]
        )
        selected = ranked.head(min(budget, len(ranked)))
        estimates.append((
            selected.label.eq("KEEP").mean(),
            selected.label.eq("KEEP").sum() / max(1, sample.label.eq("KEEP").sum()),
        ))
    estimates = np.asarray(estimates)
    return {
        "precision_low": float(np.quantile(estimates[:, 0], 0.025)),
        "precision_high": float(np.quantile(estimates[:, 0], 0.975)),
        "recall_low": float(np.quantile(estimates[:, 1], 0.025)),
        "recall_high": float(np.quantile(estimates[:, 1], 0.975)),
    }


bootstrap_rows = []
for feature in top_features[:3]:
    for budget in (50, 100):
        bootstrap_rows.append({"feature": feature, "budget": budget, **domain_cluster_bootstrap(enriched, feature, budget)})
bootstrap_table = pd.DataFrame(bootstrap_rows)
bootstrap_table.to_csv(TABLE_DIR / "domain_cluster_bootstrap_intervals.csv", index=False)
display(bootstrap_table)

## 13. Error-analysis index and reproducible decision-support summary

In [ ]:
surprising_unknown = enriched.loc[enriched.label.eq("UNKNOWN") & enriched.scored_pages.gt(0)].nlargest(
    50, "all_unique__ai_dimensions__final__top3_mean_max"
)[["company_id", "company_name", "domain", "confidence", "shared_domain", "enrichment_stratum", "scored_pages", "visible_scored_pages", "cross_host_page_share", "duplicate_page_share", "all_unique__technology__final__top3_mean", "all_unique__innovation__final__top3_mean", "all_unique__ecosystem__final__top3_mean"]]
surprising_unknown.to_csv(TABLE_DIR / "surprising_unknowns.csv", index=False)

disagreement = enriched.copy()
rank_columns = []
for dimension in DIMENSIONS:
    source = f"all_unique__{dimension}__final__top3_mean"
    rank = f"{dimension}_percentile_rank"
    disagreement[rank] = disagreement[source].rank(pct=True)
    rank_columns.append(rank)
disagreement["dimension_rank_range"] = disagreement[rank_columns].max(axis=1) - disagreement[rank_columns].min(axis=1)
disagreement.nlargest(50, "dimension_rank_range")[["company_id", "company_name", "domain", "label", "shared_domain", "dimension_rank_range", *rank_columns]].to_csv(
    TABLE_DIR / "dimension_disagreement_cases.csv", index=False
)

case_index = pd.DataFrame([
    {"table": path.name, "rows": len(pd.read_csv(path)), "path": str(path.relative_to(OUTPUT_DIR))}
    for path in sorted(TABLE_DIR.glob("*.csv"))
])
case_index.to_csv(OUTPUT_DIR / "table_index.csv", index=False)

data_quality = {
    "enriched_companies": len(enriched),
    "enriched_score_covered": int(enriched.scored_pages.gt(0).sum()),
    "enriched_no_score": int(enriched.scored_pages.eq(0).sum()),
    "enriched_shared_domains": int(enriched.shared_domain.fillna(False).sum()),
    "enriched_cross_host_companies": int(enriched.cross_host_pages.gt(0).sum()),
    "pilot_companies": len(pilot),
    "pilot_score_covered": int(pilot.scored_pages.gt(0).sum()),
    "all_lexical_signals": len(lexical),
    "resolved_lexical_signals": int(lexical.resolved_scraped_page_id.notna().sum()),
    "unresolved_lexical_signals": int(lexical.resolved_scraped_page_id.isna().sum()),
    "lexical_id_url_conflicts": int(lexical.page_id_url_conflict.sum()),
    "strict_phrase_grounded_signals": int(lexical.phrase_grounded_strict.sum()),
    "normalized_phrase_grounded_signals": int(lexical.phrase_grounded_normalized.sum()),
    "strict_context_grounded_signals": int(lexical.context_grounded_strict.sum()),
    "normalized_context_grounded_signals": int(lexical.context_grounded_normalized.sum()),
}
best_features = feature_metrics.query("formulation == 'KEEP_vs_all'").head(10)
best_budget_points = (
    budget_table.loc[budget_table.budget.isin([25, 50, 100, 125])]
    .sort_values(["budget", "keep_precision", "keep_recall"], ascending=[True, False, False])
    .groupby("budget", observed=True).head(5)
)
findings = {
    "analysis_id": ANALYSIS_ID,
    "calibration_is_not_validation": True,
    "enriched_label_counts": enriched.label.value_counts().to_dict(),
    "pilot_label_counts": pilot.label.value_counts().to_dict(),
    "data_quality": data_quality,
    "best_apparent_features": best_features.to_dict(orient="records"),
    "best_budget_points": best_budget_points.to_dict(orient="records"),
    "caveats": [
        "OpenAI labels are model judgments based on token-limited blind packages.",
        "The enriched sample cannot estimate Austrian AI prevalence.",
        "All feature and threshold comparisons are in-sample calibration diagnostics.",
        "Shared-domain and duplicate-content groups violate naive independence assumptions.",
        "The broad pilot scoring context covers only a subset of its labeled companies.",
    ],
}
(OUTPUT_DIR / "decision_support_summary.json").write_text(json.dumps(findings, ensure_ascii=False, indent=2, default=str) + "\n", encoding="utf-8")
display(pd.DataFrame([data_quality]))
display(best_features)
display(best_budget_points)
display(case_index)

## 14. Interpretation boundaries and next validation work

### Findings supported by current evidence

The executed tables and figures above describe how score distributions, repeated lexical evidence, evidence coverage, site size, duplicate content, and attribution conditions align with blind OpenAI labels in these two calibration samples. The machine-readable `decision_support_summary.json` records the strongest *apparent* company features and budget operating points without promoting them to production rules.

### Failure modes to review for scorer v1

Use the high-non-KEEP page casebooks together with grounded `false_positive`, `third_party_ai`, and `generic_marketing` phrases. Shared-domain, cross-host, duplicated-content, news/partner, and single-page-spike cases require manual inspection before any scorer-v2 change.

### Candidate scorer-v2 improvements

`candidate_vocabulary_phrases.csv` and `candidate_downweighting_phrases.csv` use normalized page-grounded phrases and remain review queues only. They must not be copied automatically into knowledge bases because OpenAI lexical suggestions can be ungrounded, context-dependent, or sample-specific.

### Candidate company-level aggregation ideas

Compare content-unique top-N summaries and repeated-evidence counts with raw maxima. Resource-constrained choices should be assessed using precision, KEEP capture, non-KEEP leakage, and fixed selection counts rather than generic accuracy alone.

### What remains uncertain

UNKNOWN is not a homogeneous negative class. Broad-pilot positive recall cannot be assessed with one KEEP. Shared domains and very large websites reduce effective independence. OpenAI labels remain fallible and evidence-package-limited. Historical `page_id` values required dual-namespace and URL resolution, while model-generated context remains less reliably verbatim than signal phrases.

### Independent validation requirement

Any scorer-v2 vocabulary, company aggregation, or operating threshold selected after this notebook must be frozen and evaluated on a future independent sample. No output of this notebook is an approved classifier, retention rule, or estimate of ecosystem prevalence.